In [1]:
import pandas as pd
import numpy as np

df_b = pd.read_csv('../data/raw/Zomato_bengaluru.csv')
df_g = pd.read_csv('../data/raw/Zomato_global.csv', encoding='latin-1')

print(f"Bengaluru raw shape : {df_b.shape}")
print(f"Global raw shape    : {df_g.shape}")

Bengaluru raw shape : (51717, 17)
Global raw shape    : (9551, 21)


In [2]:
# These columns have zero analytical value
cols_to_drop = [
    'url',          # website link
    'phone',        # contact number
    'address',      # full address (location column is enough)
    'reviews_list', # raw review text
    'menu_item',    # raw menu text
    'dish_liked'    # 54.29% null - too sparse
]

df_b.drop(columns=cols_to_drop, inplace=True)

print(f"Shape after dropping columns: {df_b.shape}")
print(f"Remaining columns: {df_b.columns.tolist()}")

Shape after dropping columns: (51717, 11)
Remaining columns: ['name', 'online_order', 'book_table', 'rate', 'votes', 'location', 'rest_type', 'cuisines', 'approx_cost(for two people)', 'listed_in(type)', 'listed_in(city)']


In [3]:
# Step 1 — Replace dirty string values with NaN
df_b['rate'] = df_b['rate'].replace(['NEW', '-'], np.nan)

# Step 2 — Remove space before /5 first
# Handles both '3.8/5' AND '3.8 /5'
df_b['rate'] = df_b['rate'].str.replace(' ', '', regex=False)

# Step 3 — Extract the numeric part before '/5'
df_b['rate'] = df_b['rate'].str.extract(r'(\d+\.\d+)').astype(float)

# Verify
print("Rate column after cleaning:")
print(df_b['rate'].describe())
print(f"\nNull count in rate: {df_b['rate'].isnull().sum()}")
print(f"\nSample values: {df_b['rate'].unique()[:10]}")

Rate column after cleaning:
count    41665.000000
mean         3.700449
std          0.440513
min          1.800000
25%          3.400000
50%          3.700000
75%          4.000000
max          4.900000
Name: rate, dtype: float64

Null count in rate: 10052

Sample values: [4.1 3.8 3.7 3.6 4.6 4.  4.2 3.9 3.1 3. ]


In [4]:
# Remove commas and convert to float
# '1,200' → '1200' → 1200.0
df_b['approx_cost(for two people)'] = (
    df_b['approx_cost(for two people)']
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Rename for cleanliness
df_b.rename(columns={
    'approx_cost(for two people)' : 'cost_for_two',
    'listed_in(type)'             : 'listed_type',
    'listed_in(city)'             : 'listed_city'
}, inplace=True)

print("Cost column after cleaning:")
print(df_b['cost_for_two'].describe())

Cost column after cleaning:
count    51371.000000
mean       555.431566
std        438.850728
min         40.000000
25%        300.000000
50%        400.000000
75%        650.000000
max       6000.000000
Name: cost_for_two, dtype: float64


In [5]:
# Encode Yes/No → 1/0
df_b['online_order'] = df_b['online_order'].map({'Yes': 1, 'No': 0})
df_b['book_table']   = df_b['book_table'].map({'Yes': 1, 'No': 0})

print("Online order distribution:")
print(df_b['online_order'].value_counts())

# Drop rows where CORE columns are null
df_b.dropna(subset=['rate', 'cost_for_two', 'cuisines', 'location'], 
            inplace=True)

# Final null check
print(f"\nFinal null counts:\n{df_b.isnull().sum()}")
print(f"\n✅ Bengaluru final shape: {df_b.shape}")

Online order distribution:
online_order
1    30444
0    21273
Name: count, dtype: int64

Final null counts:
name              0
online_order      0
book_table        0
rate              0
votes             0
location          0
rest_type       147
cuisines          0
cost_for_two      0
listed_type       0
listed_city       0
dtype: int64

✅ Bengaluru final shape: (41410, 11)


In [6]:
# Step 1 — Filter India only
df_g = df_g[df_g['Country Code'] == 1].copy()
print(f"After India filter: {df_g.shape}")

# Step 2 — Drop useless columns
cols_to_drop_g = [
    'Restaurant ID',
    'Country Code',
    'Locality Verbose',
    'Is delivering now',
    'Switch to order menu',
    'Rating color',
    'Currency'
]
df_g.drop(columns=cols_to_drop_g, inplace=True)

# Step 3 — Encode Yes/No
df_g['Has Table booking']   = df_g['Has Table booking'].map({'Yes': 1, 'No': 0})
df_g['Has Online delivery'] = df_g['Has Online delivery'].map({'Yes': 1, 'No': 0})

# Step 4 — Rename columns to match Bengaluru
df_g.rename(columns={
    'Restaurant Name'      : 'name',
    'City'                 : 'city',
    'Address'              : 'address',
    'Locality'             : 'location',
    'Cuisines'             : 'cuisines',
    'Average Cost for two' : 'cost_for_two',
    'Has Table booking'    : 'book_table',
    'Has Online delivery'  : 'online_order',
    'Price range'          : 'price_range',
    'Aggregate rating'     : 'rate',
    'Rating text'          : 'rating_text',
    'Votes'                : 'votes',
    'Longitude'            : 'longitude',
    'Latitude'             : 'latitude'
}, inplace=True)

# Step 5 — Remove unrated restaurants (rate = 0)
df_g = df_g[df_g['rate'] > 0]

# Step 6 — Drop nulls in core columns
df_g.dropna(subset=['rate', 'cuisines', 'location'], inplace=True)

print(f"\nFinal null counts:\n{df_g.isnull().sum()}")
print(f"\n✅ Global final shape: {df_g.shape}")

After India filter: (8652, 21)

Final null counts:
name            0
city            0
address         0
location        0
longitude       0
latitude        0
cuisines        0
cost_for_two    0
book_table      0
online_order    0
price_range     0
rate            0
rating_text     0
votes           0
dtype: int64

✅ Global final shape: (6513, 14)


In [12]:
import os

# Create the processed folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Now save
df_b.to_csv('../data/processed/bengaluru_cleaned.csv', index=False)
df_g.to_csv('../data/processed/global_cleaned.csv', index=False)

print("=" * 50)
print("✅ PHASE 3 COMPLETE")
print("=" * 50)
print(f"Bengaluru cleaned : {df_b.shape}")
print(f"Global cleaned    : {df_g.shape}")
print("\nFiles saved to data/processed/")

✅ PHASE 3 COMPLETE
Bengaluru cleaned : (41410, 11)
Global cleaned    : (6513, 14)

Files saved to data/processed/


In [13]:
import os

# Check if processed folder exists
print("Processed folder exists:", os.path.exists('../data/processed'))

# List all files in processed folder
print("\nFiles in processed folder:")
for f in os.listdir('../data/processed'):
    print(f"  → {f}")

# Check current working directory
print(f"\nYou are currently running from:")
print(os.getcwd())

Processed folder exists: True

Files in processed folder:
  → bengaluru_cleaned.csv
  → global_cleaned.csv

You are currently running from:
C:\Users\Yash Chavan\Data Analysis (Zomato)\notebooks
